# [GIS] SKT 좌표-행정구역 공간 조인 — 2026-09-26

- 분석 목적: 2025년 7월 SKT 성·연령 유동인구 지점을 강남구·춘천시 경계에 매핑
- 입력 데이터: `sk_통신인구_202507_통합.csv`, SGIS 2025년 2분기 시군구 경계
- 분석 기간: 2025년 7월
- 주요 파라미터: `EPSG:5179`, 공간 조건 `intersects`
- 난수 시드: 사용하지 않음
- 실행 환경: Python 3.11 이상, `uv sync --extra dev --extra gis`
- 해석 범위: 개인이 아닌 비식별 격자 지점 단위 집계이며, 경계 밖 지점은 지역 집계에서 제외


In [1]:
from pathlib import Path
import os
import geopandas as gpd
import pandas as pd
from IPython.display import display

KEYS = ["STD_YM", "BLOCK_CD", "X_COORD", "Y_COORD"]
TARGETS = ["강남구", "춘천시"]
SKT_CRS = "EPSG:5179"

repo_root = Path.cwd()
if not (repo_root / "data" / "raw").exists():
    repo_root = repo_root.parent

search_roots = [Path(os.getenv("BIGCONTEST_ROOT", repo_root)).resolve()]
local_root = Path.home() / "Desktop" / "빅콘 공모전"
if local_root.exists() and local_root not in search_roots:
    search_roots.append(local_root)

def find_one(filename):
    for root in search_roots:
        matches = list(dict.fromkeys(p.resolve() for p in root.rglob(filename)))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise FileNotFoundError(
                f"{filename}: {root} 안에서 여러 파일을 발견했습니다.\n{matches}"
            )

    raise FileNotFoundError(f"{filename}: 검색 경로에서 찾지 못했습니다. {search_roots}")

SKT_CSV = find_one("sk_통신인구_202507_통합.csv")
BOUNDARY_SHP = find_one("bnd_sigungu_00_2025_2Q.shp")
OUTPUT_DIR = repo_root / "data" / "processed" / "skt_spatial_join_202507"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("SKT:", SKT_CSV)
print("경계:", BOUNDARY_SHP)
print("출력:", OUTPUT_DIR)

SKT: /Users/leejungyeon/Workspace/projects/team-hambugi-lab/bigcontest/bigcontest-social-isolation/data/raw/sk_통신인구_202507_통합.csv
경계: /Users/leejungyeon/Workspace/projects/team-hambugi-lab/bigcontest/bigcontest-social-isolation/data/raw/국가데이터처_SGIS 행정구역 통계 및 경계/2. 경계/2. 2025년 2분기 기준 시군구 경계/bnd_sigungu_00_2025_2Q.shp
출력: /Users/leejungyeon/Workspace/projects/team-hambugi-lab/bigcontest/bigcontest-social-isolation/data/processed/skt_spatial_join_202507


## 1. 7월 분석 지점 선정

월별 통합 파일에서 12개 성·연령 지표가 모두 존재하는 행을 사용한다. 지점 수는 `STD_YM + BLOCK_CD + X_COORD + Y_COORD` 기준이며 좌표 중복도 별도로 확인한다.

In [2]:
dtype = {col: "string" for col in KEYS}
df = pd.read_csv(
    SKT_CSV, encoding="utf-8-sig", dtype=dtype, na_values=["NULL"]
)

age_cols = [
    c for c in df.columns
    if c.startswith(("MAN_FLOW_POP_CNT_", "WMAN_FLOW_POP_CNT_"))
]
if len(age_cols) != 12:
    raise ValueError(f"성·연령 지표 12개가 필요하지만 {len(age_cols)}개입니다.")

points_df = df.loc[df[age_cols].notna().all(axis=1), KEYS].copy()
key_duplicates = int(points_df.duplicated(KEYS).sum())
xy_duplicates = int(points_df.duplicated(["X_COORD", "Y_COORD"]).sum())

print(f"통합 파일 전체 행: {len(df):,}")
print(f"성·연령 데이터가 모두 있는 분석 지점: {len(points_df):,}")
print(f"결합 키 중복: {key_duplicates:,}")
print(f"좌표 중복: {xy_duplicates:,}")

if key_duplicates or xy_duplicates:
    raise ValueError("중복 지점이 있어 공간 조인을 중단합니다.")

points_df["X_NUM"] = pd.to_numeric(points_df["X_COORD"], errors="raise")
points_df["Y_NUM"] = pd.to_numeric(points_df["Y_COORD"], errors="raise")

통합 파일 전체 행: 117,825
성·연령 데이터가 모두 있는 분석 지점: 98,129
결합 키 중복: 0
좌표 중복: 0


## 2. SGIS 경계 확인

시군구 경계에서 강남구와 춘천시만 선택하고 경계 기준일과 좌표계를 기록한다.

In [3]:
boundary = gpd.read_file(BOUNDARY_SHP)
if boundary.crs is None:
    raise ValueError("경계 좌표계가 없습니다. .prj 파일을 확인하세요.")

target_boundary = boundary.loc[
    boundary["SIGUNGU_NM"].isin(TARGETS),
    ["BASE_DATE", "SIGUNGU_CD", "SIGUNGU_NM", "geometry"],
].to_crs(SKT_CRS)

if set(target_boundary["SIGUNGU_NM"]) != set(TARGETS):
    raise ValueError("강남구와 춘천시 경계를 모두 찾지 못했습니다.")

print("원본 경계 CRS:", boundary.crs)
print("분석 CRS:", target_boundary.crs)
display(target_boundary.drop(columns="geometry"))

원본 경계 CRS: EPSG:5179
분석 CRS: EPSG:5179


,BASE_DATE,SIGUNGU_CD,SIGUNGU_NM
22,20250630,11230,강남구
120,20250630,32010,춘천시


## 3. 공간 조인

`intersects`를 사용해 경계선 위의 점도 포함한다. 한 지점이 두 지역에 배정되는지도 검사한다.

In [4]:
points = gpd.GeoDataFrame(
    points_df,
    geometry=gpd.points_from_xy(points_df["X_NUM"], points_df["Y_NUM"]),
    crs=SKT_CRS,
)

joined = gpd.sjoin(
    points, target_boundary, how="left", predicate="intersects"
)
if joined.index.duplicated().any():
    raise ValueError("한 지점이 둘 이상의 대상 경계에 배정됐습니다.")

region = joined["SIGUNGU_NM"].fillna("경계 밖")
counts = region.value_counts()
mapped = int(joined["SIGUNGU_NM"].notna().sum())

summary = pd.DataFrame([
    {"구분": "전체 분석 지점", "지점수": len(joined)},
    {"구분": "강남구 경계 안", "지점수": int(counts.get("강남구", 0))},
    {"구분": "춘천시 경계 안", "지점수": int(counts.get("춘천시", 0))},
    {"구분": "두 지역 경계 밖", "지점수": int(counts.get("경계 밖", 0))},
])
summary["매핑성공률"] = pd.NA
summary.loc[summary["구분"] == "전체 분석 지점", "매핑성공률"] = mapped / len(joined)
display(summary)

,구분,지점수,매핑성공률
0,전체 분석 지점,98129,0.999368
1,강남구 경계 안,15249,<NA>
2,춘천시 경계 안,82818,<NA>
3,두 지역 경계 밖,62,<NA>


## 4. 결과 저장

경계 밖 지점은 지역 집계에서 제외하되 별도 파일로 보존한다. 최근접 경계로 강제 배정하지 않는다.

In [5]:
mapping = joined.drop(columns=["geometry", "index_right"])
outside = mapping.loc[mapping["SIGUNGU_NM"].isna()].copy()

mapping.to_csv(
    OUTPUT_DIR / "202507_SKT_시군구_매핑.csv",
    index=False, encoding="utf-8-sig", na_rep="NULL",
)
outside.to_csv(
    OUTPUT_DIR / "202507_경계밖_62개.csv",
    index=False, encoding="utf-8-sig", na_rep="NULL",
)
summary.to_csv(
    OUTPUT_DIR / "202507_공간조인_요약.csv",
    index=False, encoding="utf-8-sig",
)

assert len(joined) == 98_129
assert int(counts.get("강남구", 0)) == 15_249
assert int(counts.get("춘천시", 0)) == 82_818
assert len(outside) == 62

print("저장 완료:", OUTPUT_DIR)
print("경계 밖 처리: 지역별 집계 제외, 별도 CSV 보존, 강제 배정 없음")

저장 완료: /Users/leejungyeon/Workspace/projects/team-hambugi-lab/bigcontest/bigcontest-social-isolation/data/processed/skt_spatial_join_202507
경계 밖 처리: 지역별 집계 제외, 별도 CSV 보존, 강제 배정 없음
